# 03 · Tuning: Grid Search vs. Random Search

Implementação do slide **"Pausa para Código"** do bloco de Tuning: `GridSearchCV` e `RandomizedSearchCV` no nosso `KNNClassifier`, comparando tempo de execução e resultado final.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import time
import numpy as np
import pandas as pd

from distance_models import KNNClassifier
from datasets_utils import load_classification_dataset
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

X_train, X_test, y_train, y_test, _ = load_classification_dataset('breast_cancer')

## 1. Grid Search

Definimos uma grade de valores para `k`, `weighting` e `p`, e testamos **todas** as combinações.

In [2]:
param_grid = {
    'k': [3, 5, 7, 9, 11, 15, 21],
    'weighting': ['majority', 'distance'],
    'p': [1, 2],
}
n_combinacoes = len(param_grid['k']) * len(param_grid['weighting']) * len(param_grid['p'])
print(f'Grid Search vai testar {n_combinacoes} combinações (x 5 folds = {n_combinacoes*5} treinos)')

t0 = time.time()
grid = GridSearchCV(KNNClassifier(), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
tempo_grid = time.time() - t0

print(f'Tempo: {tempo_grid:.2f}s')
print('Melhores parâmetros:', grid.best_params_)
print('Melhor F1 (validação cruzada):', round(grid.best_score_, 4))

Grid Search vai testar 28 combinações (x 5 folds = 140 treinos)


Tempo: 0.38s
Melhores parâmetros: {'k': 3, 'p': 1, 'weighting': 'majority'}
Melhor F1 (validação cruzada): 0.9743


## 2. Random Search

Mesmo espaço de busca, mas sorteando apenas um número fixo de combinações (aqui, 8 — bem menos que as 28 do grid completo).

In [3]:
t0 = time.time()
random_search = RandomizedSearchCV(
    KNNClassifier(), param_grid, n_iter=8, cv=5, scoring='f1', random_state=42, n_jobs=-1
)
random_search.fit(X_train, y_train)
tempo_random = time.time() - t0

print(f'Tempo: {tempo_random:.2f}s')
print('Melhores parâmetros:', random_search.best_params_)
print('Melhor F1 (validação cruzada):', round(random_search.best_score_, 4))

Tempo: 0.10s
Melhores parâmetros: {'weighting': 'majority', 'p': 1, 'k': 3}
Melhor F1 (validação cruzada): 0.9743


## 3. Comparando tempo e resultado final

In [4]:
comparacao = pd.DataFrame({
    'Método': ['Grid Search', 'Random Search'],
    'Combinações testadas': [n_combinacoes, 8],
    'Tempo (s)': [round(tempo_grid, 2), round(tempo_random, 2)],
    'Melhor F1 (CV)': [round(grid.best_score_, 4), round(random_search.best_score_, 4)],
})
comparacao

,Método,Combinações testadas,Tempo (s),Melhor F1 (CV)
0,Grid Search,28,0.38,0.9743
1,Random Search,8,0.10,0.9743


In [5]:
from sklearn.metrics import f1_score

melhor_grid = grid.best_estimator_
melhor_random = random_search.best_estimator_

for nome, modelo in [('Grid', melhor_grid), ('Random', melhor_random)]:
    modelo.fit(X_train, y_train)
    f1_teste = f1_score(y_test, modelo.predict(X_test))
    print(f'{nome:8s} -> F1 no conjunto de TESTE: {f1_teste:.4f}')

Grid     -> F1 no conjunto de TESTE: 0.9640
Random   -> F1 no conjunto de TESTE: 0.9640


---
**Note:** mesmo testando bem menos combinações, o Random Search geralmente chega perto do resultado do Grid Search — essa é exatamente a discussão de Bergstra & Bengio (2012) do slide.

**Pronto para o FEMa:** basta trocar `KNNClassifier()` por `FEMaClassifier()` e ajustar `param_grid` para os hiperparâmetros da função de base escolhida — todo o resto deste notebook funciona sem alteração.